In [1]:
import pandas as pd
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.tag import pos_tag
from nltk.stem import SnowballStemmer, WordNetLemmatizer
from nltk.probability import FreqDist
from nltk.classify import NaiveBayesClassifier, accuracy
import string
import random
import pickle

In [2]:
import kagglehub
from pathlib import Path

# Downloaded once to the kagglehub cache (~/.cache/kagglehub), outside the repo
DATA_DIR = Path(kagglehub.dataset_download("lakshmi25npathi/imdb-dataset-of-50k-movie-reviews"))

C:\Users\jorda\Code\ml-notebooks\courses\natural-language-processing\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  0%|          | 0.00/25.7M [00:00<?, ?B/s]

  4%|▍         | 1.00M/25.7M [00:01<00:29, 883kB/s]

  8%|▊         | 2.00M/25.7M [00:01<00:14, 1.77MB/s]

 12%|█▏        | 3.00M/25.7M [00:01<00:08, 2.84MB/s]

 19%|█▉        | 5.00M/25.7M [00:01<00:04, 5.23MB/s]

... (8 outputs left out)

 89%|████████▉ | 23.0M/25.7M [00:02<00:00, 15.3MB/s]

 97%|█████████▋| 25.0M/25.7M [00:02<00:00, 15.6MB/s]

100%|██████████| 25.7M/25.7M [00:02<00:00, 9.04MB/s]

Extracting files...


In [3]:
# Load Dataset
dataset = pd.read_csv(DATA_DIR / 'IMDB Dataset.csv').sample(n = 800)

In [4]:
# Split Dataset
review_list = dataset['review'].tolist()
label_list = dataset['sentiment'].tolist()
dataset.head()

,review,sentiment
41951,"Cusack does his best David Niven in this one, ...",positive
33845,I am never a big fan of Taiwan movie productio...,negative
21220,"Oz is a great series, one of televisions under...",positive
28417,Rodney Dangerfield is a great. He has done a l...,negative
27924,One of the best western movies ever made. Unfo...,positive


In [5]:
# Pre-Processing
# Tokenize
word_list = []
for sent in review_list:
    words = word_tokenize(sent)
    
    for word in words:
        word_list.append(word)

print(word_list)

['Cusack', 'does', 'his', 'best', 'David', 'Niven', 'in', 'this', 'one', ',', 'although', 'I', 'do', "n't", 'know', 'if', 'anyone', 'besides', 'me', 'noticed', 'it.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'When', 'seen', 'with', 'this', 'in', 'mind', ',', 'its', 'a', 'deliciously', 'over', 'saturated', "'", 'wants', 'to', 'be', 'taken', 'more', 'seriously', 'than', 'Austin', 'powers', 'but', 'still', 'be', 'pretty', 'd', '*', 'mn', 'funny', "'", 'reworking', 'of', 'an', 'under-appreciated', 'comedy', 'classic.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'Hillary', 'Duff', 'does', 'an', 'over', 'sexualized', 'Britney', 'spears', 'lap', 'dance', 'version', 'of', 'Mata', 'Bond', '.', 'The', 'writers', 'built', 'a', 'little', 'reverse', 'Oedipus', 'twist', 'into', 'the', 'plot', '-', 'interesting', 'choice.', '<', 'br', '/', '>', '<', 'br', '/', '>', 'I', 'never', 'knew', 'how', 'soothing', 'Montel', "'s", 'voice', 'was', 'until', 'this', 'movie', '...', 'I', 'think', 'he', 'has', 'fou

In [6]:
# Remove stopwords, punctuation, numbers
eng_stopwords = stopwords.words('english')
punctuation_list = string.punctuation

In [7]:
def remove_stopwords(w_list):
    return [word for word in w_list if word not in eng_stopwords]

In [8]:
def remove_punctuation(w_list):
    return [word for word in w_list if word not in punctuation_list]

In [9]:
def remove_numbers(w_list):
    return [word for word in w_list if not word.isdigit()]

In [10]:
# Stemming, Lemmatizing
stemming = SnowballStemmer('english')
wnl = WordNetLemmatizer()

In [11]:
def stemming_word(w_list):
    return [stemming.stem(word) for word in w_list]

In [12]:
def get_tag(tag):
    if tag =='jj':
        return 'a'
    elif tag == ['nn', 'vb', 'rb']:
        return tag[0]
    else:
        return None

In [13]:
def lemmatizing_word(w_list):
    lemma = []
    tagging = pos_tag(w_list)
    # small, JJ
    
    for word, tag in tagging:
        label = get_tag(tag.lower())
        
        if label != None:
            lemma.append(wnl.lemmatize(word, label))
        else:
            lemma.append(wnl.lemmatize(word))
            
    return lemma

In [14]:
word_list = remove_stopwords(word_list)
word_list = remove_punctuation(word_list)
word_list = remove_numbers(word_list)
word_list = stemming_word(word_list)
word_list = lemmatizing_word(word_list)
word_list

['cusack',
 'best',
 'david',
 'niven',
 'one',
 'although',
 'i',
 "n't",
 'know',
 'anyon',
 'besid',
 'notic',
 'it.',
 'br',
 'br',
 'when',
 'seen',
 'mind',
 'delici',
 'satur',
 'want',
 'taken',
 'serious',
 'austin',
 'power',
 'still',
 'pretti',
 'mn',
 'funni',
 'rework',
 'under-appreci',
 'comedi',
 'classic.',
 'br',
 'br',
 'hillari',
 'duff',
 'sexual',
 'britney',
 'spear',
 'lap',
 'danc',
 'version',
 'mata',
 'bond',
 'the',
 'writer',
 'built',
 'littl',
 'revers',
 'oedipus',
 'twist',
 'plot',
 'interest',
 'choice.',
 'br',
 'br',
 'i',
 'never',
 'knew',
 'sooth',
 'montel',
 "'s",
 'voic',
 'movi',
 '...',
 'i',
 'think',
 'found',
 'new',
 'call',
 'nav',
 'system',
 'voic',
 'over',
 'i',
 'never',
 'big',
 'fan',
 'taiwan',
 'movi',
 'product',
 'oppos',
 'korean',
 'hongkong',
 'even',
 'china',
 'strong',
 'act',
 'qualiti',
 'hard',
 'found',
 'clear',
 'shown',
 'film',
 'i',
 "n't",
 'consid',
 'hard-to-pleas',
 'audienc',
 'i',
 'fact',
 'fan',
 'ind

In [15]:
labelled_data = list(zip(review_list, label_list))

features_set = []

for sentence, label in labelled_data:
    features = {}
    # CTRL + D = select variable yang sama, spam CTRL + D buat select semuanya
    check_list = word_tokenize(sentence)
    check_list = remove_stopwords(check_list)
    check_list = remove_punctuation(check_list)
    check_list = remove_numbers(check_list)
    check_list = stemming_word(check_list)
    check_list = lemmatizing_word(check_list)
    # print(check_list)
    
    for word in check_list:
        features[word] = (word in word_list)
    features_set.append((features, label))
    
features_set

[({'cusack': True,
   'best': True,
   'david': True,
   'niven': True,
   'one': True,
   'although': True,
   'i': True,
   "n't": True,
   'know': True,
   'anyon': True,
   'besid': True,
   'notic': True,
   'it.': True,
   'br': True,
   'when': True,
   'seen': True,
   'mind': True,
   'delici': True,
   'satur': True,
   'want': True,
   'taken': True,
   'serious': True,
   'austin': True,
   'power': True,
   'still': True,
   'pretti': True,
   'mn': True,
   'funni': True,
   'rework': True,
   'under-appreci': True,
   'comedi': True,
   'classic.': True,
   'hillari': True,
   'duff': True,
   'sexual': True,
   'britney': True,
   'spear': True,
   'lap': True,
   'danc': True,
   'version': True,
   'mata': True,
   'bond': True,
   'the': True,
   'writer': True,
   'built': True,
   'littl': True,
   'revers': True,
   'oedipus': True,
   'twist': True,
   'plot': True,
   'interest': True,
   'choice.': True,
   'never': True,
   'knew': True,
   'sooth': True,
   '

In [16]:
random.shuffle(features_set)
# Split train & test (80 : 20)
train_count = int(len(dataset) * 0.8)
train_dataset =  features_set[:train_count]
test_dataset = features_set[train_count:]

In [17]:
# Naive Bayes
classifier = NaiveBayesClassifier.train(train_dataset)
print(classifier.show_most_informative_features(10))
print()
print(f"Accuracy: {accuracy(classifier, test_dataset) * 100}%")

Most Informative Features


                      aw = True           negati : positi =     12.2 : 1.0
                    wast = True           negati : positi =     11.5 : 1.0
                     fit = True           positi : negati =     10.3 : 1.0
                outstand = True           positi : negati =      9.6 : 1.0
                   chick = True           negati : positi =      9.6 : 1.0
                    dull = True           negati : positi =      9.6 : 1.0
                   worst = True           negati : positi =      7.6 : 1.0
                  random = True           negati : positi =      7.5 : 1.0
                   subtl = True           positi : negati =      7.2 : 1.0
                    tear = True           positi : negati =      7.2 : 1.0
None

Accuracy: 74.375%


In [18]:
input_reviews = "This is the best movie I've ever watched"
label = classifier.classify(FreqDist(word_tokenize(input_reviews)))
print(f"Label: {label}")

Label: negative


In [19]:
from pathlib import Path

MODEL_DIR = Path.home() / ".cache" / "ml-notebooks" / "natural-language-processing/imdb-naive-bayes"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
# Save and Load Model
file = open(MODEL_DIR / "model.pickle", "wb") # Write
pickle.dump(classifier, file)
file.close()

In [20]:
file = open(MODEL_DIR / "model.pickle", "rb") # Load
classifier = pickle.load(file)
file.close()